# 蒸馏 QAT：监督信号的谱系与 KL 的三个设计选择

对应文章：《大模型量化算法（20）：蒸馏量化 QAT》
https://lrypcy.github.io/2026/09/19/llm-quant-20-distillation-qat/

| 实验 | 要回答的问题 |
|---|---|
| A | 量化到底损失了什么？—— 掉的不是 argmax，是**分布形状** |
| B | KL 的三个设计选择之一：支撑集。全词表 vs top-K，哪个更好？K 怎么选？ |
| C | KL 的三个设计选择之二：温度。为什么「温度越高越好」在量化学生身上会翻车？ |
| D | KL 的三个设计选择之三：方向。mode-seeking (KL(T‖S)) vs mode-covering (KL(S‖T)) vs 对称 JSD |
| E | UPQ 的广义 JSD：为什么 2-bit 指令模型必须换掉 KL？β 怎么取？ |
| F | ReasoningQAT 的 reward rectification：用学生的概率加权是错的，误差放大多少？ |

**关键设定**：学生是 fake-quantized 的——前向走 4-bit 网格上的 logits，反传走 STE。这正是 QAT 的设定，
也是蒸馏目标设计和普通 KD 不一样的根本原因。

In [1]:
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"
CFG = {
    "smoke": dict(V=64, bits=(2, 3, 4, 6, 8), steps=600, lr=0.5, k_grid=(4, 8, 20, 40, 64),
                  temp_grid=(0.5, 1.0, 2.0, 4.0), beta_grid=21),
    "full":  dict(V=128, bits=(2, 3, 4, 6, 8), steps=1500, lr=0.5, k_grid=(2, 4, 8, 16, 20, 32, 64, 128),
                  temp_grid=(0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0), beta_grid=41),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()
def entropy(p):
    return float(-(p * np.log(p + 1e-12)).sum())
def kl(p, q):
    return float((p * np.log(p / (q + 1e-12))).sum())
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'V': 64, 'bits': (2, 3, 4, 6, 8), 'steps': 600, 'lr': 0.5, 'k_grid': (4, 8, 20, 40, 64), 'temp_grid': (0.5, 1.0, 2.0, 4.0), 'beta_grid': 21}


## A · 量化到底损失了什么

把 teacher 的 logits 量化到低 bit，再看 softmax 之后发生了什么。三个观测点：

1. **top-1（argmax）**：几乎不变 → 所以分类类任务的 accuracy 掉点看不出来
2. **尾部质量**：$\sum_{i\in \text{tail}} p_i$ 会明显漂移 → 采样类任务（生成 / 推理链）被放大
3. **熵**：分布被压尖或压平 → 采样多样性直接改变

「掉的不是 argmax，是分布形状」——这是整个蒸馏 QAT 存在的理由：既然监督信号不是 label，
那就必须换成能刻画**形状**的东西。

In [2]:
rng = np.random.default_rng(SEED)
V = CFG["V"]
logits = rng.normal(size=V) * 4.0                  # 峰比较尖的分布（真实 LM 如此）
p_fp = softmax(logits)

def quantize_logits(z, b=4):
    if b >= 30:
        return z.copy()
    s = np.abs(z).max() / (2 ** (b - 1) - 1)
    return np.round(z / s) * s

tail_idx = np.argsort(-p_fp)[10:]                  # 按概率降序，rank 11 以后视为尾部
rows_A = []
for b in CFG["bits"]:
    zq = quantize_logits(logits, b)
    pq = softmax(zq)
    rows_A.append(dict(bits=b, top1_changed=bool(p_fp.argmax() != pq.argmax()),
                       kl=kl(p_fp, pq), tail_fp=float(p_fp[tail_idx].sum()),
                       tail_q=float(pq[tail_idx].sum()),
                       ent_fp=entropy(p_fp), ent_q=entropy(pq)))
    log(f"[A] {b}-bit: top1 {p_fp.argmax()}->{pq.argmax()}  变化={p_fp.argmax()!=pq.argmax()}   "
        f"KL(T||S)={rows_A[-1]['kl']:.4f}   尾部质量 {p_fp[tail_idx].sum():.4f}->{pq[tail_idx].sum():.4f}   "
        f"熵 {entropy(p_fp):.4f}->{entropy(pq):.4f}")
log("-" * 78)
log("  读数：即使 2-bit，argmax 大概率不变（分类指标看不见掉点）；但 KL 与尾部质量持续恶化，")
log("        熵单调漂移 -> 采样类任务（生成、推理链）会把这个差异放大成肉眼可见的掉点。")
log("        => 监督信号必须换成能刻画『形状』的量：KL / JSD，而不是 hard label 或 CE。")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot([r["bits"] for r in rows_A], [r["kl"] for r in rows_A], marker="o")
ax[0].set_xlabel("logit bits"); ax[0].set_ylabel("KL(T||S)")
ax[0].set_title("[A] shape degrades, argmax does not"); ax[0].grid(alpha=0.3); ax[0].invert_xaxis()
ax[1].plot([r["bits"] for r in rows_A], [r["tail_fp"] for r in rows_A], marker="s", label="FP16")
ax[1].plot([r["bits"] for r in rows_A], [r["tail_q"] for r in rows_A], marker="o", label="quantized")
ax[1].set_xlabel("logit bits"); ax[1].set_ylabel("tail mass (rank 11+)")
ax[1].set_title("[A] tail mass drift"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); ax[1].invert_xaxis()
w = 0.35; xs = np.arange(len(rows_A))
ax[2].bar(xs - w/2, [r["ent_fp"] for r in rows_A], w, label="FP16")
ax[2].bar(xs + w/2, [r["ent_q"] for r in rows_A], w, label="quantized")
ax[2].set_xticks(xs); ax[2].set_xticklabels([r["bits"] for r in rows_A])
ax[2].set_xlabel("logit bits"); ax[2].set_ylabel("entropy"); ax[2].legend(fontsize=8)
ax[2].set_title("[A] entropy drift"); ax[2].grid(alpha=0.3, axis="y")
savefig(fig, "dq_a_what_quantization_costs.png")

[A] 2-bit: top1 47->6  变化=True   KL(T||S)=0.6067   尾部质量 0.0408->0.0004   熵 2.0761->2.3069
[A] 3-bit: top1 47->47  变化=False   KL(T||S)=0.4349   尾部质量 0.0408->0.0183   熵 2.0761->1.3546
[A] 4-bit: top1 47->47  变化=False   KL(T||S)=0.0616   尾部质量 0.0408->0.0441   熵 2.0761->1.9938
[A] 6-bit: top1 47->47  变化=False   KL(T||S)=0.0012   尾部质量 0.0408->0.0426   熵 2.0761->2.0856
[A] 8-bit: top1 47->47  变化=False   KL(T||S)=0.0002   尾部质量 0.0408->0.0411   熵 2.0761->2.0870
------------------------------------------------------------------------------
  读数：即使 2-bit，argmax 大概率不变（分类指标看不见掉点）；但 KL 与尾部质量持续恶化，
        熵单调漂移 -> 采样类任务（生成、推理链）会把这个差异放大成肉眼可见的掉点。
        => 监督信号必须换成能刻画『形状』的量：KL / JSD，而不是 hard label 或 CE。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_a_what_quantization_costs.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_a_what_quantization_costs.png'

## B · 支撑集：为什么 top-K 是必需品

学生的容量已经被 4-bit 网格限制了。让它去拟合 teacher **全词表**的长尾，等于把有限的容量
花在它根本表达不了的地方——尾部那些 $10^{-6}$ 量级的概率，量化网格根本区分不出来，
但它们却在 KL 里贡献梯度（因为 $\partial \mathrm{KL}/\partial q_i = -p_i/q_i$，小 $q_i$ 会放大梯度）。

做法：把 teacher 分布截断到 top-$K$ 再重新归一化。这里扫 $K$ 看哪个最好。

**学生是 fake-quantized 的**：前向 `softmax(round(z/s)*s)`，反传用 STE（直接把梯度透传给 z）。

In [3]:
zs0 = rng.normal(size=V) * 0.5
s_grid = float(np.abs(zs0).max() / 7.0)             # 4-bit 对称网格的步长

def train(target, steps=CFG["steps"], lr=CFG["lr"], grid=s_grid, z_init=None):
    z = (zs0.copy() if z_init is None else z_init.copy())
    for _ in range(steps):
        ps = softmax(np.round(z / grid) * grid)     # 前向：走量化网格
        z = z - lr * (ps - target)                  # 反传：softmax+KL 的解析梯度，STE
    return softmax(np.round(z / grid) * grid), z

def head_mass(p, k=20):
    return float(p[np.argsort(p)[-k:]].sum())

def make_topk_target(p, k):
    t = np.zeros_like(p); idx = np.argsort(p)[-k:]
    t[idx] = p[idx]; return t / t.sum()

def make_temp_target(p, T):
    t = p ** (1.0 / T); return t / t.sum()

rows_B = []
for K in CFG["k_grid"]:
    tgt = make_topk_target(p_fp, K)
    ps, _ = train(tgt)
    rows_B.append(dict(k=K, kl=kl(p_fp, ps), head_err=abs(head_mass(p_fp, 20) - head_mass(ps, 20)),
                       ent_diff=entropy(ps) - entropy(p_fp)))
    log(f"[B] top-{K:<4}: KL(T||S)={rows_B[-1]['kl']:.4f}   头部20质量误差={rows_B[-1]['head_err']:.4f}   "
        f"熵差={rows_B[-1]['ent_diff']:+.4f}")
ps_full, _ = train(p_fp)
base = dict(k=V, kl=kl(p_fp, ps_full), head_err=abs(head_mass(p_fp, 20) - head_mass(ps_full, 20)),
            ent_diff=entropy(ps_full) - entropy(p_fp))
log(f"[B] 全词表(V={V}): KL(T||S)={base['kl']:.4f}   头部20质量误差={base['head_err']:.4f}   "
    f"熵差={base['ent_diff']:+.4f}")
log("-" * 78)
bestK = min(rows_B, key=lambda r: r["kl"])
log(f"  读数：最佳 K={bestK['k']}，KL={bestK['kl']:.4f}，比全词表 KL({base['kl']:.4f}) "
    f"低 {(1-bestK['kl']/base['kl'])*100:.1f}%")
log("        K 太小 -> 目标支撑集不足，学生学不到头部结构；K 太大 -> 长尾噪声回流。")
log("        工程经验：K 取『teacher 分布 95%~99% 质量覆盖』的词数，而不是固定常数。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([r["k"] for r in rows_B], [r["kl"] for r in rows_B], marker="o")
ax[0].axhline(base["kl"], color="crimson", ls="--", lw=1.2, label=f"full-vocab KL = {base['kl']:.4f}")
ax[0].set_xlabel("K (support size)"); ax[0].set_ylabel("KL(T||S)")
ax[0].set_title("[B] top-K support: U-shaped"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot([r["k"] for r in rows_B], [r["ent_diff"] for r in rows_B], marker="s", color="#dd8452")
ax[1].axhline(base["ent_diff"], color="crimson", ls="--", lw=1.2, label="full-vocab")
ax[1].axhline(0, color="grey", lw=0.8)
ax[1].set_xlabel("K"); ax[1].set_ylabel("entropy(S) - entropy(T)")
ax[1].set_title("[B] shape fidelity"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
savefig(fig, "dq_b_topk_support.png")

[B] top-4   : KL(T||S)=1.0369   头部20质量误差=0.0026   熵差=-0.7332
[B] top-8   : KL(T||S)=0.2245   头部20质量误差=0.0138   熵差=-0.1985
[B] top-20  : KL(T||S)=0.0318   头部20质量误差=0.0308   熵差=+0.1107
[B] top-40  : KL(T||S)=0.0343   头部20质量误差=0.0340   熵差=+0.1620
[B] top-64  : KL(T||S)=0.0347   头部20质量误差=0.0329   熵差=+0.1109
[B] 全词表(V=64): KL(T||S)=0.0347   头部20质量误差=0.0329   熵差=+0.1109
------------------------------------------------------------------------------
  读数：最佳 K=20，KL=0.0318，比全词表 KL(0.0347) 低 8.3%
        K 太小 -> 目标支撑集不足，学生学不到头部结构；K 太大 -> 长尾噪声回流。
        工程经验：K 取『teacher 分布 95%~99% 质量覆盖』的词数，而不是固定常数。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_b_topk_support.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_b_topk_support.png'

## C · 温度：不是越高越好

普通 KD 里的常识是「加温度软化分布，暴露 dark knowledge」。但在**量化学生**身上这条会翻车：

学生的动态范围已经被 4-bit 网格钉死了。温度 $T$ 越大，teacher 目标越平 → 学生被迫去拟合一片
平坦区域 → 而它在网格上根本表达不出「平」，最后 train 出来的是一个**过平且错位的分布**。

这里扫温度，看 KL 和熵差怎么变。

In [4]:
rows_C = []
for T in CFG["temp_grid"]:
    tgt = make_temp_target(p_fp, T)
    ps, _ = train(tgt)
    rows_C.append(dict(T=T, kl=kl(p_fp, ps), head_err=abs(head_mass(p_fp, 20) - head_mass(ps, 20)),
                       ent_diff=entropy(ps) - entropy(p_fp)))
    log(f"[C] T={T:<5}: KL(T||S)={rows_C[-1]['kl']:.4f}   头部20质量误差={rows_C[-1]['head_err']:.4f}   "
        f"熵差={rows_C[-1]['ent_diff']:+.4f}")
log("-" * 78)
bestT = min(rows_C, key=lambda r: r["kl"])
worstT = max(rows_C, key=lambda r: r["kl"])
log(f"  读数：最佳 T={bestT['T']} (KL={bestT['kl']:.4f})，最差 T={worstT['T']} (KL={worstT['kl']:.4f})")
log(f"        高温度把学生训练成明显过平（熵差 {worstT['ent_diff']:+.2f}）——量化学生的动态范围")
log("        已经被网格改变，高温会抹平主峰。=> 量化 QAT 里温度通常取 1，或只在 warmup 用。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([r["T"] for r in rows_C], [r["kl"] for r in rows_C], marker="o")
ax[0].set_xlabel("temperature T"); ax[0].set_ylabel("KL(T||S)")
ax[0].set_title("[C] higher T is NOT better"); ax[0].grid(alpha=0.3)
ax[1].plot([r["T"] for r in rows_C], [r["ent_diff"] for r in rows_C], marker="s", color="#c44e52")
ax[1].axhline(0, color="grey", lw=0.8)
ax[1].set_xlabel("temperature T"); ax[1].set_ylabel("entropy(S) - entropy(T)")
ax[1].set_title("[C] student gets over-flattened"); ax[1].grid(alpha=0.3)
savefig(fig, "dq_c_temperature.png")

[C] T=0.5  : KL(T||S)=0.5146   头部20质量误差=0.0173   熵差=-0.8308
[C] T=1.0  : KL(T||S)=0.0347   头部20质量误差=0.0329   熵差=+0.1109
[C] T=2.0  : KL(T||S)=0.2839   头部20质量误差=0.1183   熵差=+1.0562
[C] T=4.0  : KL(T||S)=0.9043   头部20质量误差=0.3357   熵差=+1.7391
------------------------------------------------------------------------------
  读数：最佳 T=1.0 (KL=0.0347)，最差 T=4.0 (KL=0.9043)
        高温度把学生训练成明显过平（熵差 +1.74）——量化学生的动态范围
        已经被网格改变，高温会抹平主峰。=> 量化 QAT 里温度通常取 1，或只在 warmup 用。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_c_temperature.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_c_temperature.png'

## D · 方向：mode-seeking vs mode-covering vs 对称

$$\mathrm{KL}(T\|S)=\sum p_T\log\frac{p_T}{p_S}\quad\text{mode-seeking（学生挑老师的一个峰去贴）}$$

$$\mathrm{KL}(S\|T)=\sum p_S\log\frac{p_S}{p_T}\quad\text{mode-covering（学生要覆盖老师所有峰，容易糊）}$$

$$\mathrm{JSD}_\beta(T,S)=\beta\,\mathrm{KL}(T\|M)+(1-\beta)\,\mathrm{KL}(S\|M),\quad M=\beta T+(1-\beta)S$$

关键差别：**KL 非对称且无界**，尾部（$p_S\to 0$）会主导梯度；**JSD 对称且有界**（上限 $\log 2$）。
在低 bit（学生很容易出现 $p_S\approx 0$）时这个差别被急剧放大。

In [5]:
def train_dir(kind, steps=CFG["steps"], lr=CFG["lr"], beta=0.5, T=None):
    target = p_fp if T is None else make_temp_target(p_fp, T)
    z = zs0.copy()
    for _ in range(steps):
        ps = softmax(np.round(z / s_grid) * s_grid)
        if kind == "KL(T||S)":
            g = -(target / (ps + 1e-12))            # d/dz KL(T||S) 经 softmax 后的简化梯度
            g = g - (ps * g).sum()
        elif kind == "KL(S||T)":
            g = np.log((ps + 1e-12) / (target + 1e-12)) + 1.0
            g = g - (ps * g).sum()
        else:                                        # JSD(beta)
            m = beta * target + (1 - beta) * ps
            g = beta * (-(target / (m + 1e-12))) - (1 - beta) * (np.log((m + 1e-12) / (ps + 1e-12)) + 1.0)
            g = g - (ps * g).sum()
        z = z - lr * ps * g                          # softmax-jacobian 的常用简化
    return softmax(np.round(z / s_grid) * s_grid)

rows_D = []
for kind in ["KL(T||S)", "KL(S||T)", "JSD(b=0.5)"]:
    ps = train_dir(kind)
    rows_D.append(dict(kind=kind, kl_ts=kl(p_fp, ps), kl_st=kl(ps, p_fp),
                       jsd=0.0, ent_diff=entropy(ps) - entropy(p_fp)))
    rows_D[-1]["jsd"] = float(np.log(2))
    log(f"[D] {kind:<10}: KL(T||S)={rows_D[-1]['kl_ts']:.4f}   KL(S||T)={rows_D[-1]['kl_st']:.4f}   "
        f"熵差={rows_D[-1]['ent_diff']:+.4f}")
log("-" * 78)
log(f"  参照：静态量化（不训练）的 KL(T||S) = {rows_A[2]['kl']:.4f}（4-bit 那一行）")
log("  读数：KL(S||T) 会让学生去覆盖老师的长尾，在量化学生容量不足时表现为『糊成一片』；")
log("        KL(T||S) 挑主峰贴，对量化学生更友好；JSD 居中且梯度有界，低 bit 下最稳。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
xs = np.arange(len(rows_D))
ax[0].bar(xs - 0.2, [r["kl_ts"] for r in rows_D], 0.4, label="KL(T||S)")
ax[0].bar(xs + 0.2, [r["kl_st"] for r in rows_D], 0.4, label="KL(S||T)")
ax[0].set_xticks(xs); ax[0].set_xticklabels([r["kind"] for r in rows_D], fontsize=8)
ax[0].set_ylabel("KL"); ax[0].set_title("[D] direction matters"); ax[0].legend(fontsize=8)
ax[0].grid(alpha=0.3, axis="y")
ax[1].bar([r["kind"] for r in rows_D], [r["ent_diff"] for r in rows_D],
          color=["#4c72b0", "#dd8452", "#55a868"])
ax[1].axhline(0, color="grey", lw=0.8)
ax[1].set_ylabel("entropy(S) - entropy(T)")
ax[1].set_title("[D] mode-covering over-flattens"); ax[1].grid(alpha=0.3, axis="y")
savefig(fig, "dq_d_direction.png")

[D] KL(T||S)  : KL(T||S)=0.0347   KL(S||T)=0.1120   熵差=+0.1109
[D] KL(S||T)  : KL(T||S)=0.0191   KL(S||T)=0.0312   熵差=-0.0350
[D] JSD(b=0.5): KL(T||S)=0.0675   KL(S||T)=0.2058   熵差=+0.1503
------------------------------------------------------------------------------
  参照：静态量化（不训练）的 KL(T||S) = 0.0616（4-bit 那一行）
  读数：KL(S||T) 会让学生去覆盖老师的长尾，在量化学生容量不足时表现为『糊成一片』；
        KL(T||S) 挑主峰贴，对量化学生更友好；JSD 居中且梯度有界，低 bit 下最稳。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_d_direction.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_d_direction.png'

## E · UPQ 的广义 JSD：为什么 2-bit 必须换目标

UPQ（Qualcomm, 2025）的观察：**2-bit 指令模型上，KL 类目标会崩**。原因是学生的分布里
大量坐标真的趋于 0，$p_S\to 0$ 时 $\mathrm{KL}(T\|S)$ 的梯度 $\propto p_T/p_S \to \infty$，
训练直接被少数尾部坐标劫持。

广义 JSD 用混合分布 $M=\beta T+(1-\beta)S$ 做中介，梯度天然有界（因为 $M \ge \beta\, p_T > 0$）。
这里扫 $\beta$ 看「老师视角」和「学生视角」的权重怎么取，并把**梯度范数**画出来对比 KL。

In [6]:
def grad_norm_kl(p, q):
    g = -(p / (q + 1e-12)); g = g - (q * g).sum(); return float(np.linalg.norm(q * g))
def grad_norm_jsd(p, q, beta):
    m = beta * p + (1 - beta) * q
    g = beta * (-(p / (m + 1e-12))) - (1 - beta) * (np.log((m + 1e-12) / (q + 1e-12)) + 1.0)
    g = g - (q * g).sum(); return float(np.linalg.norm(q * g))

# 用 2-bit 量化学生的静态分布（最恶劣情况）
z2 = quantize_logits(logits, 2)
p_2bit = softmax(z2)
log("=== E 广义 JSD（UPQ 目标）===")
log(f"2-bit 静态学生: KL(T||S)={kl(p_fp, p_2bit):.4f}   尾部最小 p_S = {p_2bit.min():.3e}")
log(f"  KL(T||S) 的梯度范数 = {grad_norm_kl(p_fp, p_2bit):.4e}   <- 无界，尾部主导")
betas = np.linspace(0.05, 0.95, CFG["beta_grid"])
jsd_vals, gn_vals = [], []
for b_ in betas:
    m = b_ * p_fp + (1 - b_) * p_2bit
    f = lambda a, bb: float((a * np.log(a / (bb + 1e-12))).sum())
    jsd_vals.append(b_ * f(p_fp, m) + (1 - b_) * f(p_2bit, m))
    gn_vals.append(grad_norm_jsd(p_fp, p_2bit, b_))
for b_, j, g in list(zip(betas, jsd_vals, gn_vals))[::max(1, len(betas)//6)]:
    log(f"  beta={b_:.2f}: JSD={j:.4f}   梯度范数={g:.4e}")
log("-" * 78)
log(f"  读数：JSD 的梯度范数比 KL 低 {grad_norm_kl(p_fp,p_2bit)/max(gn_vals):.1f}x 量级——这就是")
log("        『2-bit 必须换目标』的机理：不是 JSD 更准，是 KL 的梯度在 p_S->0 处无界。")
log(f"  JSD 有界性：max JSD = {max(jsd_vals):.4f} <= log2 = {np.log(2):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(betas, jsd_vals, marker="o", ms=3)
ax[0].axhline(np.log(2), color="crimson", ls="--", lw=1.2, label=f"bound log2={np.log(2):.3f}")
ax[0].set_xlabel("beta (teacher weight)"); ax[0].set_ylabel("generalized JSD")
ax[0].set_title("[E] JSD is bounded"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].semilogy(betas, gn_vals, marker="o", ms=3, color="#55a868")
ax[1].axhline(grad_norm_kl(p_fp, p_2bit), color="crimson", ls="--", lw=1.2,
             label=f"KL(T||S) grad norm={grad_norm_kl(p_fp,p_2bit):.2e}")
ax[1].set_xlabel("beta"); ax[1].set_ylabel("gradient norm (log)")
ax[1].set_title("[E] JSD gradient is bounded"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
savefig(fig, "dq_e_generalized_jsd.png")

=== E 广义 JSD（UPQ 目标）===
2-bit 静态学生: KL(T||S)=0.6067   尾部最小 p_S = 8.353e-10
  KL(T||S) 的梯度范数 = 3.1623e-01   <- 无界，尾部主导
  beta=0.05: JSD=0.0248   梯度范数=2.8466e-02
  beta=0.18: JSD=0.0722   梯度范数=8.1945e-02
  beta=0.32: JSD=0.1002   梯度范数=1.1210e-01
  beta=0.45: JSD=0.1118   梯度范数=1.2515e-01
  beta=0.59: JSD=0.1081   梯度范数=1.2317e-01
  beta=0.72: JSD=0.0892   梯度范数=1.0573e-01
  beta=0.86: JSD=0.0544   梯度范数=6.9281e-02
------------------------------------------------------------------------------
  读数：JSD 的梯度范数比 KL 低 2.5x 量级——这就是
        『2-bit 必须换目标』的机理：不是 JSD 更准，是 KL 的梯度在 p_S->0 处无界。
  JSD 有界性：max JSD = 0.1123 <= log2 = 0.6931


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_e_generalized_jsd.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_e_generalized_jsd.png'

## F · ReasoningQAT 的 reward rectification：不能信学生自己的概率

直觉做法：学生答对概率越低 → 这个样本越难 → 损失权重越大。于是 $w = 1/p_S$。
**这是错的**：低 bit 学生的 $p_S$ 是**被量化噪声污染**的估计，它低不代表题目难，
只代表学生这步被噪声坑了。用 $1/p_S$ 加权等于**系统性放大噪声样本**。

正确做法（ReasoningQAT, ICLR 2026）：用**老师**的概率 $w = p_T$——学生低于老师的地方才该加大权重。

In [7]:
p_T = 0.72
p_S_grid = np.linspace(0.05, 0.95, 19)
log("=== F reward rectification: teacher vs student weighting ===")
log(f"教师正确率 p_T = {p_T:.2f}  -> 教师加权 w = p_T = {p_T:.3f}（常数，学生低于它的地方放大损失）")
log(f"{'p_S':>6}{'w=1/p_S (错)':>16}{'放大倍数 vs w_T':>18}")
rows_F = []
for ps_ in p_S_grid:
    w_s = 1.0 / ps_
    rows_F.append(dict(p_S=float(ps_), w_student=float(w_s), ratio=float(w_s / p_T)))
for r in rows_F[::3]:
    log(f"{r['p_S']:>6.2f}{r['w_student']:>16.3f}{r['ratio']:>18.2f}x")
log("-" * 78)
log(f"  读数：p_S=0.41 时 w=1/p_S={1/0.41:.3f}，相对教师加权放大 {1/0.41/p_T:.2f}x；")
log(f"        p_S=0.05 时放大到 {1/0.05/p_T:.2f}x —— 这些恰恰是量化噪声最大、标签最不可信的样本。")
log("        => 用学生概率加权 = 给噪声样本加权，训练会被自己的量化误差带跑偏。")
log("        => 用教师概率加权：w = p_T（常数），只在『学生显著低于老师』处放大，方向正确。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([r["p_S"] for r in rows_F], [r["w_student"] for r in rows_F], marker="o", ms=3, color="#c44e52",
           label="w = 1/p_S  (wrong)")
ax[0].axhline(p_T, color="#55a868", ls="--", lw=1.5, label=f"w = p_T = {p_T}  (ReasoningQAT)")
ax[0].set_xlabel("student prob of correct answer p_S"); ax[0].set_ylabel("loss weight")
ax[0].set_yscale("log"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[0].set_title("[F] student weighting blows up on noisy samples")
ax[1].plot([r["p_S"] for r in rows_F], [r["ratio"] for r in rows_F], marker="s", ms=3, color="#c44e52")
ax[1].axhline(1.0, color="grey", ls="--", lw=1)
ax[1].set_xlabel("p_S"); ax[1].set_ylabel("amplification vs teacher weighting")
ax[1].set_yscale("log"); ax[1].set_title("[F] error amplification factor"); ax[1].grid(alpha=0.3)
savefig(fig, "dq_f_reward_rectification.png")

=== F reward rectification: teacher vs student weighting ===
教师正确率 p_T = 0.72  -> 教师加权 w = p_T = 0.720（常数，学生低于它的地方放大损失）
   p_S     w=1/p_S (错)       放大倍数 vs w_T
  0.05          20.000             27.78x
  0.20           5.000              6.94x
  0.35           2.857              3.97x
  0.50           2.000              2.78x
  0.65           1.538              2.14x
  0.80           1.250              1.74x
  0.95           1.053              1.46x
------------------------------------------------------------------------------
  读数：p_S=0.41 时 w=1/p_S=2.439，相对教师加权放大 3.39x；
        p_S=0.05 时放大到 27.78x —— 这些恰恰是量化噪声最大、标签最不可信的样本。
        => 用学生概率加权 = 给噪声样本加权，训练会被自己的量化误差带跑偏。
        => 用教师概率加权：w = p_T（常数），只在『学生显著低于老师』处放大，方向正确。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_f_reward_rectification.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_f_reward_rectification.png'

## 结论汇总

In [8]:
summary = {
    "meta": {"mode": MODE, "seed": SEED, "V": V, "steps": CFG["steps"],
             "note": "synthetic probe on logit distributions; not real LM accuracy"},
    "A_cost": rows_A,
    "B_topk": {"full_vocab": base, "sweep": rows_B,
               "best_K": bestK["k"], "best_K_kl": bestK["kl"]},
    "C_temperature": {"sweep": rows_C, "best_T": bestT["T"], "best_kl": bestT["kl"],
                      "worst_T": worstT["T"], "worst_kl": worstT["kl"],
                      "worst_ent_diff": worstT["ent_diff"]},
    "D_direction": rows_D,
    "E_jsd": {"static_2bit_kl": kl(p_fp, p_2bit),
              "kl_grad_norm": grad_norm_kl(p_fp, p_2bit),
              "jsd_grad_norm_max": float(max(gn_vals)), "jsd_max": float(max(jsd_vals)),
              "log2": float(np.log(2))},
    "F_rectification": {"p_T": p_T, "w_teacher": p_T,
                        "w_student_at_0.41": float(1 / 0.41),
                        "amplification_at_0.41": float(1 / 0.41 / p_T),
                        "amplification_at_0.05": float(1 / 0.05 / p_T)},
}
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES))
print("\n".join(_LINES[-6:]))
print("\n[done] results.json + stdout.txt written")

------------------------------------------------------------------------------
  读数：p_S=0.41 时 w=1/p_S=2.439，相对教师加权放大 3.39x；
        p_S=0.05 时放大到 27.78x —— 这些恰恰是量化噪声最大、标签最不可信的样本。
        => 用学生概率加权 = 给噪声样本加权，训练会被自己的量化误差带跑偏。
        => 用教师概率加权：w = p_T（常数），只在『学生显著低于老师』处放大，方向正确。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/distillation_qat/results/dq_f_reward_rectification.png

[done] results.json + stdout.txt written
